In [ ]:
import astropy.units as u
from astropy.coordinates import SkyCoord

from astroquery.mast import MastMissions, Observations, Catalogs
from astropy.table import Table
from mast_aladin import MastTable, AppSidecar, MastAladin


def missions_mast(mission, env):
    missions = MastMissions(mission=mission)

    if env in ['dev', 'test', 'int']:
        base_url = f'https://mast{env}.stsci.edu'
        missions._service_api_connection.SERVICE_URL = base_url
        missions._service_api_connection.REQUEST_URL = f'{base_url}/search/{mission}/api/v0.1/'
        missions._service_api_connection.MISSIONS_DOWNLOAD_URL = f'{base_url}/search/'

    return missions


def table_selection(msg):
    #print(f'Table selection: {msg}')
    print(type(msg.type))
    if msg.type == 'change':
        mt = msg.owner

        # selected_keys = [r[mt.item_key] for r in mt.selected_rows_table]
        # print(selected_keys)
        print(mt.selected_rows)
    else:
        print(f'mast_table_obs unexpected {msg.type=}')


def object_hovered(data):
    print(f'object_hovered: {data}')


def object_clicked(data):
    print(f'object_clicked: {data}')


def click(data):
    print(f'click: {data}')


def select(data):
    print(f'select:')
    print(aladin._selected_objects)


def current_overlays(data):
    print(f'current_overlays: {data}')

def add_aladin_listeners(aladin):
    aladin.set_listener('object_hovered', object_hovered)
    aladin.set_listener('object_clicked', object_clicked)
    aladin.set_listener('click', click)
    aladin.set_listener('select', select)
    aladin.set_listener('current_overlays', current_overlays)
    


# Get obs data ready
mast = missions_mast('roman', 'test')
target = SkyCoord(ra=270, dec=66, unit='deg')
radius = 0.3 * u.deg
# obs = mast.query_region(target, radius=radius) #, select_cols=['s_region'])
obs = Table.read("roman_obs.csv")
obs_no_fp = Table(obs)
obs_no_fp.remove_column('s_region')

# Get catalog data ready
# gaia_sources = Catalogs.query_region(target, radius=0.2, catalog="Gaia", version=2)
gaia_sources = Table.read("gaia_sources.csv")



In [ ]:
aladin = MastAladin(
    target=f'{target.ra.deg} {target.dec.deg}',
    fov=0.5
)
apps_initialized = AppSidecar.open(aladin, anchor='split-right')
AppSidecar.resize_all(800)

add_aladin_listeners(aladin)

mast_table_obs = MastTable(obs)

mast_table_obs.row_select_callbacks.clear()
mast_table_obs.row_select_callbacks.append(table_selection)

mast_table_obs

In [ ]:
add_aladin_listeners(aladin)

In [ ]:
aladin.target = '270 66'

In [ ]:
aladin.objects_to_select = [
    {
        'mast_item_key': mast_table_obs.item_key,
        'sources': mast_table_obs.selected_rows
    }
]

In [ ]:
mast_table_obs.selected_rows

In [ ]:
obs_layer = aladin.add_table(obs, name='Roman Observations', color='#FF6600', 
                             ra_field="ra_ref", dec_field="dec_ref",
                            mast_item_key=mast_table_obs.item_key)

In [ ]:
aladin.remove_overlay(obs_layer)

In [ ]:
obs_no_fp_layer = aladin.add_table(obs_no_fp, name='Roman Observations NO FPs', 
                                   color='#FF6600', ra_field="ra_ref", dec_field="dec_ref",
                                  mast_item_key=mast_table_obs_no_fp.item_key)

In [ ]:
mast_table_gaia = MastTable(gaia_sources)
mast_table_gaia

In [ ]:
gaia_layer = aladin.add_table(gaia_sources)

In [ ]:
#aladin.selected_objects
mast_table_obs_no_fp.selected_rows

In [ ]:
aladin.remove_overlay(gaia_layer)

In [ ]:
aladin.listener_callback['click']

In [ ]:
aladin._overlays

In [ ]:
aladin.get_overlays()

In [ ]:
mast_table_obs_no_fp = MastTable(obs_no_fp, app=aladin)

mast_table_obs_no_fp.row_select_callbacks.clear()
mast_table_obs_no_fp.row_select_callbacks.append(table_selection)

mast_table_obs_no_fp

In [ ]:
mast_table_obs_no_fp.item_key